# 4-Amaliy Mashg'ulot: Ma’lumotlarni tozalash, yetishmayotgan qiymatlarni aniqlash va qayta ishlash
**Fan:** Data Mining  
**Mavzu:** «Do‘kon mijozlari» ma'lumotlar to'plami asosida sifat muammolarini (Missing, Duplicates, Format, Outlier) aniqlash va bartaraf etish.

In [ ]:
# 1. Kutubxonalarni import qilish va boshlang'ich datasetni yuklash
import os
import pandas as pd
import numpy as np

df = pd.read_csv('mijozlar.csv')
print('Boshlang\'ich dataset hajmi:', df.shape)
display(df)

## 1-topshiriq. Ma'lumotlar sifatini tekshirish
Aniqlangan sifat muammolari jadvali:
| Muammo turi | Qaysi atribut(lar)da? | Qaysi satr(lar)da (ID)? | Soni |
| :--- | :--- | :--- | :--- |
| **Yetishmayotgan qiymat** | Yoshi, Shahar, Oylik_xarid, Xaridlar_soni | ID: 2, 4, 5, 6, 9 | 5 ta katak |
| **Takroriy yozuv (Dublikat)** | Barcha ustunlar (Rustam) | ID: 7 (ID: 3 ning takrori) | 1 ta satr |
| **Nomuvofiq format** | Jinsi, Shahar (kichik harflar) | ID: 3 ('erkak', 'samarqand') | 2 ta ustun |
| **G'ayrioddiy qiymat (Outlier)** | Oylik_xarid (5000 ming so'm) | ID: 9 (Ulug'bek) | 1 ta qiymat |

> **Savol javobi:** Kompyuter 7-satr 3-satrning takrori ekanini darhol aniqlay olmaydi, chunki registrlar har xil ('samarqand' != 'Samarqand').

## 2-topshiriq. Yetishmayotgan qiymatlarni qayta ishlash usulini tanlash
| Atribut | Tanlangan usul | Hisoblash | To'ldiriladigan qiymat | Asoslash |
| :--- | :--- | :--- | :--- | :--- |
| `Yoshi` | Mediana | Mavjud yoshlar: [25, 27, 28, 30, 33, 34, 41] | **30 yosh** | Simmetrik taqsimot, butun son |
| `Shahar` | Moda | Samarqand: 4 ta, Toshkent: 2 ta, Buxoro: 2 ta | **Samarqand** | Kategorial ustun uchun faqat moda |
| `Oylik_xarid` | Mediana | Mavjud: [380, 450, 470, 490, 520, 560, 610, 5000] | **505 ming so'm** | Outlier (5000) ta'siriga chidamli |
| `Xaridlar_soni`| Mediana/Moda | Mavjud: [3, 4, 5, 5, 5, 6, 6, 7] | **5 marta** | Eng ko'p uchragan odatiy xaridlar soni |

In [ ]:
# Oylik xarid bo'yicha o'rtacha va mediana solishtirmasi
xarid_vals = df[df['ID'] != 7]['Oylik_xarid'].dropna()
print(f'Arifmetik o\'rtacha (Mean): {xarid_vals.mean():.2f} ming so\'m')
print(f'Mediana (Median): {xarid_vals.median():.2f} ming so\'m')
print(f'Farq: {xarid_vals.mean() - xarid_vals.median():.2f} ming so\'m (Outlier 5000 sababli o\'rtacha buzilgan)')

## 4-topshiriq. Python (pandas) yordamida tozalash jarayoni

In [ ]:
# 1. Bo'sh qiymatlarni ko'rish
print('Boshlang\'ich bo\'sh qiymatlar:')
display(df.isnull().sum())

# 2. Formatni bir xil qilish (Harflarni standartlashtirish)
for col in ['Jinsi', 'Shahar']:
    df[col] = df[col].str.strip().str.capitalize()

# 3. Takrorlarni aniqlash va o'chirish (ID ustunisiz)
ustunlar = df.columns[1:]
print('Formatlashdan so\'ng aniqlangan takrorlar:', df.duplicated(subset=ustunlar).sum())
df = df.drop_duplicates(subset=ustunlar)

# 4. Yetishmayotgan qiymatlarni to'ldirish
df['Yoshi'] = df['Yoshi'].fillna(df['Yoshi'].median())
df['Shahar'] = df['Shahar'].fillna(df['Shahar'].mode()[0])

# 5. Outlierni (5000) medianaga almashtirish
med = df['Oylik_xarid'].median()
df.loc[df['Oylik_xarid'] > 2000, 'Oylik_xarid'] = med
df['Oylik_xarid'] = df['Oylik_xarid'].fillna(med)
df['Xaridlar_soni'] = df['Xaridlar_soni'].fillna(df['Xaridlar_soni'].median())

print('\nTozalangandan keyingi bo\'sh qiymatlar soni:')
display(df.isnull().sum())
print('\nYakuniy tozalangan jadval o\'lchami:', df.shape)
display(df)